### What Happens when Correlations Break Down?

Some simple simulations regarding factor covariances
___

Paul Grajzl

Friday, November 13



In [ ]:
import sys
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from src import simulation, plotting, factors

In [14]:
prices, returns, params = simulation.simulate_random_returns()

In [15]:
simulation.plot_simulated_stock(prices, params, "S0001")

### Part 1: Simulated Prices and Returns (Independent Stocks)

A baseline world with **no market, no factors, no structure**: every stock is an independent random walk.

**Model:** geometric Brownian motion. Each day, each stock's log return is

$$
\log\frac{P_{i,t}}{P_{i,t-1}} \sim \mathcal{N}\Big(\big(\mu_i - \tfrac{1}{2}\sigma_i^2\big)\Delta t,\ \sigma_i^2 \, \Delta t\Big), \qquad \Delta t = \tfrac{1}{252}
$$

The −½σ² term makes μ the expected annual simple return. Prices start at 100 and compound.

**Setup:**
- 1,000 stocks, 5 years of business days
- Each stock's true drift μ drawn uniformly from −10% to +20% a year (about a third negative)
- Each stock's true vol σ drawn uniformly from 15–60% a year
- Fixed random seed for reproducibility; true parameters stored for comparison

**Checks:**
- Realized vol matches true vol almost exactly (correlation ~0.99): 5 years of daily data pins down vol precisely
- Average pairwise correlation between stocks ≈ 0: no common driver
- Realized drift matches true drift **poorly**: 5 years is far too short to estimate expected returns. Risk is much easier to estimate than return.

In [18]:
caps, market, w0 = simulation.simulate_market_caps(prices)

# Top 10 stocks account for roughly 35% of the market concentration (real US market: top 10 roughly 30-40%)

simulation.plot_simulated_market(market, w0)

### Part 2: Market Caps and a Simulated Market Index

**Initial caps:** each stock gets a random share of a market worth 100 at the start, drawn from a **lognormal** distribution (skew parameter 1.75). This gives a right-skewed market like the real one: most stocks small, a handful very large. **Top 10 stocks ≈ 35% of initial market cap**, in line with the real US market (~30–40%).

**Dynamics:** share counts are fixed, so each stock's cap moves with its own price:

$$
\text{cap}_{i,t} = \text{cap}_{i,0} \times \frac{P_{i,t}}{P_{i,0}}
$$

**Market index:** the sum of all caps, starting at exactly 100. Its daily return is the cap-weighted average stock return, with weights drifting as stocks rise and fall, like a real index.

**Key observation:** the simulated market is far smoother than individual stocks. With independent stocks, their moves largely cancel out, so index vol comes out well below single-stock vols (15–60%). Real indices run ~15–20% vol because stocks share a common driver: the gap a market factor will close.

In [19]:
simulation.save_sim("baseline", prices=prices, returns=returns, params=params,
                    caps=caps, market=market, w0=w0)

In [ ]:
book = simulation.simulate_book_equity(caps)
bp = book / caps

In [23]:
# 4. One stock: daily drift with price, steps at each reporting date
simulation.plot_simulated_value(caps, book, "S0001")

In [24]:
simulation.save_sim("baseline", book=book)

### Part 3: Simulated Book Equity and Value

Value = **book-to-price** = book equity / market cap. In real life it moves for two reasons, on two clocks, and the simulation reproduces both:

1. **Price moves daily**: when a stock falls, its B/P rises (it gets cheaper), and vice versa
2. **Book updates quarterly, with a lag**: book grows with retained earnings, but the market only learns the new number when the company files, weeks after quarter end. Between filings, reported book is flat (a staircase)

**Construction (fake but realistic book equity):**
- **Starting B/P**: lognormal around a median of 0.4 (price-to-book ≈ 2.5, typical US market), wide dispersion (log sd 0.8), so both deep-value and expensive growth stocks exist
- **Correlation with size**: starting log B/P correlated with starting log market cap via a knob (default **−0.3**: big companies tend to be more expensive)
- **Book growth**: each company has its own long-run growth rate (mean 8% a year, sd 6%) plus quarterly noise (3%)
- **Point-in-time reporting**: each quarter's book becomes known only after a stock-specific lag of 20–45 days, on staggered fiscal calendars, so the cross-section doesn't all update on the same day

**Checks:**
- Median starting B/P ≈ 0.4; starting value–size correlation ≈ −0.3, by construction
- **Value–momentum correlation emerges on its own** (negative), never imposed: stocks that fell over the past year have low momentum *and* high B/P, because price sits in both. Same mechanism as in real markets
- Individual B/P paths wiggle daily with price and jump ~4 times a year at reporting dates

**Knobs for later experiments**: size correlation, valuation dispersion, book growth mean and dispersion.

**Known simplification**: book growth is unrelated to stock returns. In reality, earnings surprises move prices too; this link can be added deliberately if we want value to carry a genuine premium.

In [25]:
mkt_ret = market.pct_change()

expo_raw = {
    "Size": np.log(caps),
    "Value": np.log(book / caps),
    "Momentum": factors.momentum(prices),
    "Beta": factors.beta_exposure(returns, mkt_ret),
}
styles_sim = {k: factors.standardize(v, weights=caps) for k, v in expo_raw.items()}

In [26]:
res_sim, resid_sim = factors.cs_regression_multi(
    returns, styles_sim, weights=np.sqrt(caps), return_resid=True
)
res_sim["r2"].describe()

count    1007.000000
mean        0.016600
std         0.013973
min         0.000303
25%         0.007026
50%         0.012918
75%         0.021982
max         0.140246
Name: r2, dtype: float64

In [27]:
ones = pd.DataFrame(1.0, index=returns.index, columns=returns.columns)
expo_attr = {"intercept": ones, **styles_sim}

s1 = factors.attribute("S0001", returns, expo_attr, res_sim, resid_sim).rename(columns={"intercept": "Market"})

(s1.drop(columns="Total").sum(axis=1) - s1["Total"]).abs().max()     # exactness check, ~1e-16

plotting.plot_attribution(s1, "S0001")

In [28]:
plotting.plot_table(factors.risk_decomp(s1), title="S0001: risk decomposition (no-factor world)",
                    index_name="Component")

### Why It's All Idio in the Random-Walk World

### What "idio" means

A factor model explains a stock's return using only what it **shares with other stocks**:

$$
r_{i,t} = \underbrace{\sum_k x_{ik} \, f_{k,t}}_{\text{shared}} + \underbrace{\varepsilon_{i,t}}_{\text{idio}}
$$

The factor returns $f_{k,t}$ are estimated from the **whole cross-section** each day: they describe how groups of stocks moved together. Idio is whatever part of a stock's move **nobody else shared**.

### How this world was built

Every stock's daily return was drawn **independently**:

$$
r_{i,t} = \mu_i \Delta t + \sigma_i \sqrt{\Delta t} \, z_{i,t}, \qquad z_{i,t} \text{ independent across stocks}
$$

S0001's random shock on any day has nothing to do with any other stock's shock. So there is **nothing shared** for a factor to capture: by definition, every bit of its movement is idio.

### Why each factor fails

**Size, value, momentum, beta:** the characteristics exist (some stocks are big, some cheap), but returns were never generated from them. Each day the regression asks "did big stocks beat small ones today?" The answer is pure chance: a little positive one day, a little negative the next. So the estimated factor returns are noise, unrelated to S0001's actual move.

**Market:** the market here is just the cap-weighted average of independent stocks. S0001 is one of 1,000, with a small weight $w$, so its influence on the market is tiny. Its correlation with the market is roughly

$$
\rho \approx \frac{w \, \sigma_{\text{S0001}}}{\sigma_{\text{market}}}
$$

which is close to zero for a small weight. The market moves for reasons that are all about *other* stocks.

### Why contributions exist but don't count

The regression always outputs some factor return, so S0001's exposure × factor return is never exactly zero: its contributions have real-looking vol (6–7%). But risk share depends on **co-movement with S0001's total return**:

$$
\text{share}_j = \frac{\mathrm{Cov}(c_j, r)}{\mathrm{Var}(r)}
$$

Since those contributions are noise unrelated to S0001's moves, the covariance ≈ 0 and the share ≈ 0. All variance falls to idio.

### The takeaway

Factors explain a stock only to the extent its moves are **shared with other stocks that have similar characteristics**. Independent stocks share nothing, so everything is idio. Real stocks share a lot (the market, sectors, styles), which is why real idio shares were ~75% for Apple rather than ~100%.

___

### Building Returns from Factors: Before vs Now

### Before: returns first, characteristics after

Each stock's daily return was drawn **independently** (a random walk). Characteristics were computed **afterwards** from the resulting prices:

**random returns → prices → caps, momentum, B/P, beta**

The characteristics described the stocks but had **no effect** on returns. Nothing was shared across stocks, so a factor model found nothing: ~100% idio.

### Now: characteristics drive returns, returns update characteristics

Each day, the world steps forward in a loop:

1. **Compute today's exposures from history** (as of yesterday's close)
   - Size: log market cap
   - Value: log(reported book / market cap)
   - Momentum: 12-1 month return
   - Standardized cross-sectionally, cap-weighted centered (same recipe as the real model)
2. **Draw today's factor returns** from the factor covariance matrix F
3. **Draw today's idio shocks**, independent per stock
4. **Build each stock's return**:
$$
r_{i,t} = f^{\text{mkt}}_t + \sum_k z_{ik,t-1} \, f_{k,t} + \varepsilon_{i,t}
$$
5. **Update** prices, market caps, B/P and the market index, then repeat

**Feedback loop**: a stock that rallies becomes bigger (size), more expensive (lower B/P), and eventually a momentum winner, which changes how it responds to tomorrow's factor returns. This is how real markets work: characteristics and returns co-evolve.

**Burn-in**: momentum needs a year of price history, so the first year runs with market + idio only; style factors switch on once every exposure can be computed.

### Why it matters

- **Before**: factors were descriptions. **Now**: factors are causes.
- Stocks with similar characteristics now move together, so a factor model should find real shared structure and a much smaller idio share.
- The factor covariance F is now a lever: changing it changes how stocks co-move, which is what the correlation-break experiments will exploit.

In [34]:
sw = simulation.simulate_structural_market()
returns_s, caps_s, Fs = sw["returns"], sw["caps"], sw["factor_returns"]
expo_s = sw["exposures"]

res_s, resid_s = factors.cs_regression_multi(returns_s, expo_s, weights=np.sqrt(caps_s), return_resid=True)
true = Fs.loc[res_s.index]

# 1. Recovery of the true factor returns
recovery = pd.Series({"Market": res_s["intercept"].corr(true["Market"]),
                      **{k: res_s[k].corr(true[k]) for k in expo_s}}).round(3)
print("1. Correlation of estimated vs true factor returns (after burn-in)")
print("   Expect ~0.97+ if the regression recovers what was planted\n")
print(recovery.to_string(), "\n")

# 2. Cross-sectional R²
r2 = res_s["r2"] * 100
print("2. Daily cross-sectional R² (%)")
print(f"   Mean {r2.mean():.2f}   Median {r2.median():.2f}   Max {r2.max():.2f}")
print("   Placebo floor (no factor structure): ~0.5%\n")

# 3. Emergent value-momentum exposure correlation
vm = expo_s["Value"].corrwith(expo_s["Momentum"], axis=1).dropna().mean()
print("3. Average cross-sectional correlation, value vs momentum exposure")
print(f"   {vm:.3f}   (never imposed; emerges because price sits in both)\n")

simulation.save_sim("structural", prices=sw["prices"], returns=returns_s, caps=caps_s,
                    market=sw["market"], book=sw["book"], factor_returns=Fs)
print("Saved to data/sim/structural/")

1. Correlation of estimated vs true factor returns (after burn-in)
   Expect ~0.97+ if the regression recovers what was planted

Market      0.973
Size        0.789
Value       0.911
Momentum    0.955 

2. Daily cross-sectional R² (%)
   Mean 6.92   Median 5.20   Max 39.63
   Placebo floor (no factor structure): ~0.5%

3. Average cross-sectional correlation, value vs momentum exposure
   -0.317   (never imposed; emerges because price sits in both)

Saved to data/sim/structural/


In [35]:
# Size recovery: √cap-weighted vs equal-weighted cross-sectional regression
res_ew = factors.cs_regression_multi(returns_s, expo_s)

recovery = pd.DataFrame({
    "√cap weighted": [res_s["intercept"].corr(true["Market"])] + [res_s[k].corr(true[k]) for k in expo_s],
    "Equal weighted": [res_ew["intercept"].corr(true["Market"])] + [res_ew[k].corr(true[k]) for k in expo_s],
}, index=["Market", *expo_s]).round(3)

print("Correlation of estimated vs true factor returns")
print("(higher = model recovers the planted factor more precisely)\n")
print(recovery.to_string())
print(f"\nSize: {recovery.loc['Size', '√cap weighted']:.3f} (√cap) -> "
      f"{recovery.loc['Size', 'Equal weighted']:.3f} (equal)")

Correlation of estimated vs true factor returns
(higher = model recovers the planted factor more precisely)

          √cap weighted  Equal weighted
Market            0.973           0.991
Size              0.789           0.945
Value             0.911           0.959
Momentum          0.955           0.982

Size: 0.789 (√cap) -> 0.945 (equal)


### Regression Weighting: √cap vs Equal (Simulated World)

**Correlation of estimated vs true factor returns:**

| Factor | √cap weighted | Equal weighted |
|---|---|---|
| Market | 0.973 | 0.991 |
| Size | 0.789 | **0.945** |
| Value | 0.911 | 0.959 |
| Momentum | 0.955 | 0.982 |

**Size gains most.** A slope is identified by **spread** in the regressor. √cap weighting leans on big stocks, which all have similar (high) size exposures, so the stocks that count most offer little variation in size to learn from. Size also has the smallest true vol (3%), so its signal is weakest relative to idio noise.

**Every factor improves with equal weights here.** Regression theory (GLS) says the most precise weights are **1 / idio variance**: noisier stocks should count less. In this simulation, idio vol is drawn at random, **unrelated to size**, so a big stock is no less noisy than a small one. √cap weighting down-weights small stocks for no reason and throws away information.

**Why real models use √cap anyway:** in real markets, idio vol **falls with size**: small caps are noisier and have worse data. √cap is a practical proxy for 1 / idio variance, down-weighting exactly the stocks that really are noisier.

**Takeaway:** a weighting scheme is only "right" relative to how noise relates to the stocks. Match the weights to the noise structure: equal weights when noise is uniform, √cap (or 1 / idio variance) when smaller stocks are noisier.

In [37]:
from IPython.display import display

t = "S0007"

# Estimated decomposition (equal-weighted cross-sectional regression)
res_ew, resid_ew = factors.cs_regression_multi(returns_s, expo_s, return_resid=True)
ones = pd.DataFrame(1.0, index=returns_s.index, columns=returns_s.columns)
est = factors.attribute(t, returns_s, {"intercept": ones, **expo_s}, res_ew, resid_ew)
est = est.rename(columns={"intercept": "Market"}).dropna()
idx = est.index

# True decomposition (what we planted)
Ft = Fs.loc[idx]
true_c = pd.DataFrame({
    "Market": Ft["Market"],
    **{k: expo_s[k][t].shift(1).reindex(idx) * Ft[k] for k in expo_s},
    "Idio": sw["idio"][t].reindex(idx),
})
true_c["Total"] = returns_s[t].reindex(idx)

print(f"{t}: return decomposition, structural world ({idx.min():%b %Y} to {idx.max():%b %Y})\n")
print(f"Exactness check (estimated pieces sum to total): max error {(est.drop(columns='Total').sum(axis=1) - est['Total']).abs().max():.1e}\n")

print("Latest exposures (z-scores):")
print(pd.Series({k: expo_s[k][t].dropna().iloc[-1] for k in expo_s}).round(2).to_string(), "\n")

print("Cumulative contribution (%), estimated vs true:")
print(pd.DataFrame({"Estimated": est.sum() * 100, "True": true_c.sum() * 100}).round(1).to_string(), "\n")

print("Share of variance (%), estimated vs true:")
print(pd.DataFrame({"Estimated": factors.risk_decomp(est)["Share of variance (%)"],
                    "True": factors.risk_decomp(true_c)["Share of variance (%)"]}).round(1).to_string(), "\n")

print(f"True idio vol for {t}: {sw['idio_vol'][t]:.1%} per yr   "
      f"Estimated idio vol: {est['Idio'].std() * np.sqrt(252):.1%} per yr")

display(plotting.plot_attribution(est, t))

S0007: return decomposition, structural world (Jan 2023 to Nov 2026)

Exactness check (estimated pieces sum to total): max error 3.5e-18

Latest exposures (z-scores):
Size       -2.30
Value       0.76
Momentum    0.17 

Cumulative contribution (%), estimated vs true:
          Estimated  True
Market         47.1  50.8
Size            9.6  -1.3
Value           0.8   4.6
Momentum       11.4  12.5
Idio            3.5   5.8
Total          72.4  72.4 

Share of variance (%), estimated vs true:
          Estimated   True
Market         40.0   40.2
Size            9.5    9.1
Value           3.6    3.7
Momentum        1.8    1.8
Idio           45.1   45.2
Total         100.0  100.0 

True idio vol for S0007: 17.1% per yr   Estimated idio vol: 17.2% per yr


### Why Return Attribution Is Noisy but Risk Attribution Isn't

**Result for S0007 (structural simulation, true decomposition known):**
- **Risk shares**: estimated vs true match within a few tenths of a point for every component (market 40.0 vs 40.2, idio 45.1 vs 45.2); idio vol 17.2% vs 17.1% true
- **Cumulative returns**: totals match (72.4) but the pieces are shuffled. Size: **+9.6 estimated vs −1.3 true**, offset by lower market, value and idio

**Why returns go wrong while risk doesn't:**

1. **Daily errors accumulate in sums but cancel in variances.** Each day's estimated factor return is slightly off. Variance is driven by the big daily swings, which the model gets right, so small errors barely matter. A cumulative return is the **sum** of ~1,000 small daily errors, which drifts. Same lesson as the first simulation: realized vol matched the truth almost exactly, realized drift didn't.

2. **Extreme exposures amplify errors.** A contribution is exposure × factor return, so any error in the size factor is multiplied by S0007's exposure of −2.3. Its 11-point error corresponds to only ~4–5 points of cumulative error in the size factor itself over almost four years.

3. **Size is the hardest factor to estimate** (lowest recovery correlation), so that's where the drift shows up most.

**Takeaway:** risk attribution of a position is robust; return attribution is noisy, especially for small factors and stocks with extreme exposures. "This name made 10% from size" carries a lot of estimation noise; "size is 9% of this name's risk" is far more trustworthy.

In [38]:
simulation.compare_decomposition(sw, res_ew, resid_ew, "S0044")

S0044: return decomposition, structural world (Jan 2023 to Nov 2026)

Exactness check (estimated pieces sum to total): max error 6.9e-18

Latest exposures (z-scores):
Size       -1.13
Value      -0.17
Momentum   -0.47 

Cumulative contribution (%), estimated vs true:
          Estimated   True
Market         47.1   50.8
Size            5.6   -0.6
Value          -0.6   -0.8
Momentum       42.5   42.8
Idio           22.9   25.4
Total         117.6  117.6 

Share of variance (%), estimated vs true:
          Estimated   True
Market         22.4   22.5
Size            1.0    1.0
Value           0.1    0.1
Momentum        3.3    3.2
Idio           73.2   73.3
Total         100.0  100.0 

True idio vol for S0044: 28.6% per year   Estimated idio vol: 29.5% per year


In [41]:
simulation.position_stats(sw, res_ew, resid_ew, "S0044")

### What Idio Sharpe Should We Target?

### In the simulation

The true idio Sharpe for every stock is **exactly zero**: no edge was planted, so each idio stream is pure noise. S0044's realized idio Sharpe of 0.19 is luck. With ~3.8 years of data, a Sharpe's standard error is ~1/√3.8 ≈ 0.5, so anything from roughly −1 to +1 is consistent with zero edge.

### In the real world: think in books, not names

A high Sharpe on any single name isn't necessary. What matters is the **book**, and idio diversifies. Grinold's **fundamental law of active management**:

$$
\text{book idio Sharpe} \approx \text{per-name idio Sharpe} \times \sqrt{N}
$$

for $N$ independent, similarly sized bets.

| Per-name idio Sharpe | 10 names | 30 names | 60 names |
|---|---|---|---|
| 0.2 | 0.6 | 1.1 | 1.5 |
| 0.3 | 0.9 | 1.6 | 2.3 |
| 0.5 | 1.6 | 2.7 | 3.9 |

A per-name idio Sharpe of 0.2–0.3 looks unimpressive alone but builds a strong book when repeated across enough genuinely independent names. Pod-level book idio Sharpe expectations are commonly cited around **1.5–2+** (a ballpark that varies by strategy and before costs).

### Why √N isn't a free lunch

- **Independence**: if names share a hidden driver (a crowded theme, an unmodeled factor), their idio isn't truly independent and the √N benefit collapses
- **Skill vs luck**: a single name's realized idio Sharpe over a few years is almost all noise; skill only becomes visible in aggregate, across many names and long periods
- **Costs and sizing**: trading costs, short borrow costs and uneven position sizes all reduce the book number

### Next experiment

Plant a small true edge (e.g. per-name idio Sharpe 0.3) in a subset of stocks, then test whether it's detectable **name by name** (mostly not) versus in a **diversified book** (much more so).

In [43]:
simulation.plot_simulated_stock(sw["prices"], "S0350", idio_vol=sw["idio_vol"], default="5y")

In [44]:
simulation.compare_decomposition(sw, res_ew, resid_ew, "S0350")

S0350: return decomposition, structural world (Jan 2023 to Nov 2026)

Exactness check (estimated pieces sum to total): max error 6.9e-18

Latest exposures (z-scores):
Size       -1.88
Value       1.79
Momentum   -0.21 

Cumulative contribution (%), estimated vs true:
          Estimated  True
Market         47.1  50.8
Size            5.3  -1.7
Value           5.7  11.9
Momentum      -27.4 -29.4
Idio            3.7   2.8
Total          34.4  34.4 

Share of variance (%), estimated vs true:
          Estimated   True
Market         29.8   30.0
Size            2.8    2.7
Value           7.7    7.4
Momentum        7.5    7.2
Idio           52.2   52.7
Total         100.0  100.0 

True idio vol for S0350: 21.2% per year   Estimated idio vol: 22.1% per year


In [45]:
simulation.position_stats(sw, res_ew, resid_ew, "S0350")

In [46]:
simulation.plot_simulated_stock(sw["prices"], "S0777", idio_vol=sw["idio_vol"], default="5y")

In [47]:
from IPython.display import display

display(simulation.compare_decomposition(sw, res_ew, resid_ew, "S0777"))
simulation.position_stats(sw, res_ew, resid_ew, "S0777")

S0777: return decomposition, structural world (Jan 2023 to Nov 2026)

Exactness check (estimated pieces sum to total): max error 3.5e-18

Latest exposures (z-scores):
Size       -2.91
Value       0.62
Momentum   -1.25 

Cumulative contribution (%), estimated vs true:
          Estimated  True
Market         47.1  50.8
Size            8.6  -2.5
Value           5.0   4.4
Momentum      -19.8 -19.4
Idio          -14.3  -6.7
Total          26.6  26.6 

Share of variance (%), estimated vs true:
          Estimated   True
Market         40.2   40.6
Size           10.6   10.2
Value           0.8    0.7
Momentum        7.7    7.4
Idio           40.7   41.1
Total         100.0  100.0 

True idio vol for S0777: 15.8% per year   Estimated idio vol: 16.4% per year


### Making Money vs Adding Value: S0777

**Holding S0777 (Jan 2023 – Nov 2026):** total unhedged return **6.66% per year**, true idio **−1.68% per year**, so factor PnL ≈ **+8.3% per year**.

$$
\text{total} = \text{factor PnL} + \text{idio PnL}
$$

**Cumulative contributions (%), true decomposition:**

| Component | True | Estimated |
|---|---|---|
| Market | +50.8 | +47.1 |
| Size | −2.5 | +8.6 |
| Value | +4.4 | +5.0 |
| Momentum | −19.4 | −19.8 |
| **Factor total** | **+33.3** | **+40.9** |
| Idio | −6.7 | −14.3 |
| **Total** | **+26.6** | **+26.6** |

**What happened:** S0777 rose because its **exposures** rose. The market tide (+50.8) did the heavy lifting, partly offset by its negative momentum exposure (z = −1.25), which cost it −19.4 as momentum paid off. Its own stock-specific part **lost** money (−6.7).

**Intuition:** a cheap replica of S0777's factor exposures (market ETF plus style tilts) would have earned ~8.3% per year. Holding S0777 earned 6.66%. The part that's actually *about S0777* was a drag: **you made money, but less than its exposures alone would have given you.**

**How a pod sees it:**
- **Factor PnL (~+8.3% per year)**: not credited, since the firm can buy that exposure almost for free
- **Idio PnL (−1.68% per year true)**: what the PM is judged on; here, a *loss* of stock-picking value despite the position going up

The reverse also happens: a stock can fall in a crashing market yet show **positive** idio, if it fell less than its exposures implied, and the PM gets credited despite losing money.

**Estimation note:** risk shares match the truth closely (idio 40.7% estimated vs 41.1% true; idio vol 16.4% vs 15.8%), but return attribution doesn't: the model credits size with +8.6 vs a true −2.5, and the 11-point error lands in idio (−14.3 vs −6.7). S0777's extreme size exposure (z = −2.91) amplifies size-factor estimation error, the same pattern as S0007.

**Reminder:** in this world S0777's true idio is luck, since no edge was planted, and −1.68% per year is well within one standard error of zero.

In [48]:
names = ["Size", "Value", "Momentum"]
true = Fs.loc[res_ew.index]

# 1. Recovery of the true factor returns (equal-weighted)
recovery = pd.Series({"Market": res_ew["intercept"].corr(true["Market"]),
                      **{k: res_ew[k].corr(true[k]) for k in names}}).round(3)
print("1. Correlation of estimated vs true factor returns (equal-weighted regression)\n")
print(recovery.to_string(), "\n")

# 2. Cross-sectional R²
r2 = res_ew["r2"] * 100
print("2. Daily cross-sectional R² (%), equal-weighted")
print(f"   Mean {r2.mean():.2f}   Median {r2.median():.2f}   Max {r2.max():.2f}\n")

# 3. Exposure correlations: average daily cross-sectional correlation (independent of regression weights)
expo_corr = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        expo_corr.loc[a, b] = expo_s[a].corrwith(expo_s[b], axis=1).dropna().mean()
print("3. Exposure correlations (average across days; same for any regression weighting)\n")
print(expo_corr.round(3).to_string(), "\n")

# 4. Factor-return correlations: estimated (EW) vs true (planted)
print("4. Factor-return correlations, estimated (equal-weighted)\n")
print(res_ew[names].corr().round(3).to_string(), "\n")
print("   True (planted)\n")
print(true[names].corr().round(3).to_string())

1. Correlation of estimated vs true factor returns (equal-weighted regression)

Market      0.991
Size        0.945
Value       0.959
Momentum    0.982 

2. Daily cross-sectional R² (%), equal-weighted
   Mean 6.33   Median 4.66   Max 37.57

3. Exposure correlations (average across days; same for any regression weighting)

           Size  Value  Momentum
Size      1.000 -0.399     0.143
Value    -0.399  1.000    -0.317
Momentum  0.143 -0.317     1.000 

4. Factor-return correlations, estimated (equal-weighted)

           Size  Value  Momentum
Size      1.000  -0.01     0.016
Value    -0.010   1.00    -0.260
Momentum  0.016  -0.26     1.000 

   True (planted)

           Size  Value  Momentum
Size      1.000 -0.041     0.010
Value    -0.041  1.000    -0.273
Momentum  0.010 -0.273     1.000


### How the Simulated Factor Returns Work

**Rules (planted once, fixed for the whole period):** each factor's mean and vol, and the correlations between factors, i.e. the factor covariance matrix F, never change. E.g. market 7% mean and 16% vol; value–momentum correlation −0.3, every day.

**Outcomes (drawn fresh every day):** each day's actual factor returns are a new random draw from that same fixed distribution. The market factor might be +1.2% one day and −0.8% the next, but always drawn from the same 16% vol distribution.

So factor returns are **consistent in distribution, not in realized values**: the factor process is **stationary**.

**What does change over time:**
- **Exposures**, recomputed daily from each stock's evolving price, size and book
- **Style factors switch on after a one-year burn-in**: their returns are drawn from day one, but only affect stock returns once momentum can be computed

**Factor recovery** measures how well the daily cross-sectional regression, which sees only that day's stock returns and exposures, reproduces the **true** daily factor returns: the correlation between estimated and planted series over time. It differs from **R²**, which measures fit to stock returns. R² is only ~5% because idio dominates each stock's move, yet recovery is 0.95+, because across 1,000 stocks the idio noise averages out.

**Next:** the correlation-break experiments switch F to a different regime partway through, breaking stationarity, to see what happens to risk estimates, hedges and idio.

___

### Real Factor Processes Are Not Stationary

The simulation draws factor returns from one fixed distribution forever. Real markets don't work that way:

**1. Volatility clusters.** Calm and turbulent periods come in long stretches: a big move today makes another big move tomorrow more likely. Market vol can sit near 10% for a year, then jump above 60% within weeks (2008, March 2020). A fixed vol is a long-run average, not what any given day looks like.

**2. Correlations change, especially in crises.** Stock correlations tend to rise sharply in sell-offs, just when diversification is needed most. Factor correlations shift too: value and momentum are usually negatively correlated, but the strength varies a lot over time.

**3. Factors crash.** Momentum earned steadily for years, then lost heavily within months in 2009 when beaten-down stocks rebounded violently. A stationary model with fixed vol essentially can't produce that.

**4. Premia drift and decay.** Published anomalies tend to weaken as investors trade on them (McLean and Pontiff). Value's long drought from the late 2000s through 2020 shows a premium that looked stable for decades and then wasn't.

**5. The market itself changes.** Passive investing, quant crowding, market structure and shifting sector composition slowly move the "rules."

### How risk models cope

- **Exponential weighting**: recent data dominates estimates
- **Short half-lives for vol** (fast to react), longer for correlations (more stable)
- **Regime adjustments**: scale forecasts up when recent realized risk exceeds predictions
- **Stress tests and scenarios**: for jumps a statistical model can't anticipate

**Working assumption: local stationarity.** The world is roughly stable over weeks to months, not over years. A single clean regime break in the simulation makes the cost of assuming stationarity visible.

___

### Some Factor Philosophy
___

### Why Do Factors Exist at All?

Two different questions hide in "why factors exist": why stocks **move together** (factors as risk), and why some characteristics **earn a premium** (factors as return). A factor can have the first without the second: sectors drive co-movement but earn no reliable premium.

### Why stocks move together

Companies share economic exposures. Banks all depend on interest rates and credit, energy stocks on oil, small caps on financing conditions, every stock on the economy and the price of risk. Thousands of stocks are driven by a handful of common forces, so their joint behavior has **low-dimensional structure**. Factors are the language for that structure: a compression of thousands of return series into a few shared drivers plus independent noise.

### Why some factors earn a premium

**1. Risk compensation (rational view).** A premium is payment for bearing risk that can't be diversified away and tends to hurt in bad times. Value stocks are often distressed firms that suffer most in recessions; small caps are illiquid and fragile in crises. Investors demand extra return to hold them, so the premium is a fair price, not a free lunch.

**2. Investor behavior (behavioral view).** Systematic mistakes create predictable mispricing. Momentum reflects **underreaction**: news gets priced in slowly. Value reflects **overreaction**: investors extrapolate past growth too far, overpricing glamour stocks and underpricing boring ones. Arbitrage is limited (capital, risk, career incentives), so the mispricing persists.

**3. Constraints and institutions (structural view).** Many investors can't or won't use leverage, so they buy high-beta stocks to get more market exposure, overpricing them; that's one explanation for the low-volatility anomaly. Benchmarking, flows and mandates create similar persistent distortions.

### The skeptical view

Hundreds of "factors" have been published (the **factor zoo**). With enough researchers searching the same data, many findings are statistical flukes (Harvey, Liu and Zhu). Real factors should have an economic story, survive out of sample and across markets, and not disappear the moment they're published.

### The practitioner's view

For risk management it doesn't matter **why** a factor exists, only that it drives co-movement. For alpha it matters a lot: a premium with a durable reason (risk or constraints) is more likely to persist than one that's purely a historical accident.